# Usage of PySpark SQL

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                     .appName("Analyzing an unknown article.")
                     .getOrCreate())


In [2]:
## documentation
spark.read??

In [3]:
file_path = r'article-example.txt' # fill in

In [4]:
article = spark.read.text(file_path)

In [5]:
article.show(5, truncate=False)

+-----------------------------------------------------------------------------------+
|value                                                                              |
+-----------------------------------------------------------------------------------+
|Here's a look inside security at the White House Correspondents' Association Dinner|
|By Tamara Keith                                                                    |
|                                                                                   |
|Updated Monday, April 27, 2026 • 3:30 PM EDT                                       |
|                                                                                   |
+-----------------------------------------------------------------------------------+
only showing top 5 rows


In [6]:
from pyspark.sql.functions import col

In [7]:
# All possible ways to select a column
article.select(article.value)
article.select(article['value'])
article.select(col('value'))
article.select('value')

DataFrame[value: string]

In [8]:
from pyspark.sql.functions import col, split

lines = article.select(split(col('value'), ' ').alias('line'))
lines.show(5, truncate=False)

+------------------------------------------------------------------------------------------------+
|line                                                                                            |
+------------------------------------------------------------------------------------------------+
|[Here's, a, look, inside, security, at, the, White, House, Correspondents', Association, Dinner]|
|[By, Tamara, Keith]                                                                             |
|[]                                                                                              |
|[Updated, Monday,, April, 27,, 2026, •, 3:30, PM, EDT]                                          |
|[]                                                                                              |
+------------------------------------------------------------------------------------------------+
only showing top 5 rows


In [9]:
from pyspark.sql.functions import explode

words = lines.select(explode(col('line')).alias('word'))
words.show(5)

+--------+
|    word|
+--------+
|  Here's|
|       a|
|    look|
|  inside|
|security|
+--------+
only showing top 5 rows


In [10]:
from pyspark.sql.functions import lower

words_lower = words.select(lower(col('word')).alias('word_lower'))
words_lower.show(5)

+----------+
|word_lower|
+----------+
|    here's|
|         a|
|      look|
|    inside|
|  security|
+----------+
only showing top 5 rows


In [11]:
from pyspark.sql.functions import regexp_extract

word_clean = words_lower.select(regexp_extract(col('word_lower'), r'(\W+)?([a-z]+)', 2).alias('word'))
word_clean.show(5)

+--------+
|    word|
+--------+
|    here|
|       a|
|    look|
|  inside|
|security|
+--------+
only showing top 5 rows


In [18]:
# remove words with 3 or fewer letters and report top 5 words
from pyspark.sql.functions import length

word_long = word_clean.filter(length(col('word')) > 3)
word_freqs = word_long.groupBy('word').count().orderBy('count', ascending=False)
word_freqs.show(5)

+---------+-----+
|     word|count|
+---------+-----+
|president|   29|
|     that|   25|
| security|   19|
|     this|   19|
|     with|   16|
+---------+-----+
only showing top 5 rows


In [22]:
# Report table of word lengths and their occurrence freqencies
word_len = word_long.groupBy(length(col('word')).alias('word_len')).count().orderBy('word_len')
word_len.show()

+--------+-----+
|word_len|count|
+--------+-----+
|       4|  274|
|       5|  166|
|       6|  130|
|       7|   98|
|       8|   79|
|       9|   74|
|      10|   19|
|      11|   12|
|      12|    7|
|      13|    4|
|      14|    5|
|      15|    1|
+--------+-----+



In [31]:
# make a histogram of the occurences of each alphabet
words_split = word_long.select(split(col('word'), '').alias('words_split'))
chars = words_split.select(explode(col('words_split')).alias('char'))
char_counts = chars.groupBy('char').count().orderBy('count', ascending=False)
char_counts.show(5)

char_counts.plot(kind='bar', x='char', y='count')

+----+-----+
|char|count|
+----+-----+
|   e|  716|
|   t|  485|
|   i|  398|
|   n|  378|
|   r|  377|
+----+-----+
only showing top 5 rows
